# BERTimbau — experimento inicial

Este notebook realiza o primeiro experimento com fine-tuning do
BERTimbau Base para classificação da clareza das respostas.

Antes de executar a validação cruzada completa, é realizado um experimento
piloto utilizando apenas o fold 0.

Objetivos do piloto:

- verificar a compatibilidade do treinamento com a GPU disponível;
- medir o consumo máximo de VRAM;
- estimar o tempo necessário por fold;
- verificar o funcionamento do pipeline de fine-tuning;
- obter uma primeira estimativa de desempenho.

A configuração inicial utiliza `max_length=256`, escolhido como ponto de
partida por oferecer menor custo computacional, deixando a comparação com
384 e 512 tokens para experimentos posteriores.

In [88]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [89]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [90]:
from src.evaluation.folds import carregar_folds

from src.models.bertimbau import (
    ID_TO_LABEL,
    criar_modelo,
    criar_tokenizer,
    preparar_dataset,
)

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM total:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )

PyTorch: 2.14.0+cu130
CUDA disponível: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM total: 4.0 GB


In [91]:
SEED = 42

set_seed(SEED)

In [92]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train.xlsx"
FOLDS_PATH = PROJECT_ROOT / "data" / "splits" / "folds.csv"

df = pd.read_excel(
    DATA_PATH,
    sheet_name="train",
)

df["resp_text"] = df["resp_text"].astype(str)

df = carregar_folds(
    df,
    FOLDS_PATH,
)

df.head()

,resp_text,clarity,fold
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5,3
1,"Prezada cidadã, As informações sobre óbitos ...",c1,3
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1,4
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234,4
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234,0


In [93]:
FOLD_PILOTO = 0

treino = df[df["fold"] != FOLD_PILOTO].copy()
validacao = df[df["fold"] == FOLD_PILOTO].copy()

print("Treino:", len(treino))
print("Validação:", len(validacao))

Treino: 16073
Validação: 4019


In [94]:
MAX_LENGTH = 256

LEARNING_RATE = 2e-5

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

GRADIENT_ACCUMULATION_STEPS = 2

WEIGHT_DECAY = 0.01

FP16 = True
GRADIENT_CHECKPOINTING = False

MAX_STEPS_BENCHMARK = 100

SEED = 42

In [95]:
tokenizer = criar_tokenizer()

In [96]:
dataset_treino = preparar_dataset(
    treino["resp_text"],
    treino["clarity"],
    tokenizer,
    max_length=MAX_LENGTH,
)

dataset_validacao = preparar_dataset(
    validacao["resp_text"],
    validacao["clarity"],
    tokenizer,
    max_length=MAX_LENGTH,
)

Map:   0%|          | 0/16073 [00:00<?, ? examples/s]

Map:   0%|          | 0/4019 [00:00<?, ? examples/s]

In [97]:
print(dataset_treino)
print(dataset_validacao)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16073
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4019
})


In [98]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
)

In [99]:
def calcular_metricas(eval_pred):
    logits, labels = eval_pred

    predicoes = np.argmax(
        logits,
        axis=-1,
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predicoes,
        ),
        "f1_macro": f1_score(
            labels,
            predicoes,
            average="macro",
        ),
    }

In [100]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [101]:
import gc

globals().pop("trainer", None)
globals().pop("modelo", None)
globals().pop("trainer_benchmark", None)
globals().pop("modelo_benchmark", None)

gc.collect()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(
    "VRAM alocada antes de carregar o novo modelo:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

VRAM alocada antes de carregar o novo modelo: 0.02 GB


In [102]:
modelo_benchmark = criar_modelo()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [103]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "checkpoints"
    / "bertimbau_piloto_fold0"
)

In [104]:
OUTPUT_DIR_BENCHMARK = (
    PROJECT_ROOT
    / "checkpoints"
    / "bertimbau_benchmark"
)

training_args_benchmark = TrainingArguments(
    output_dir=str(OUTPUT_DIR_BENCHMARK),

    learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS_BENCHMARK,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    weight_decay=WEIGHT_DECAY,

    fp16=FP16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,

    eval_strategy="no",
    save_strategy="no",

    logging_strategy="steps",
    logging_steps=20,

    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

In [105]:
trainer_benchmark = Trainer(
    model=modelo_benchmark,
    args=training_args_benchmark,
    train_dataset=dataset_treino,
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [106]:
print(len(dataset_treino))
print(len(dataset_validacao))

16073
4019


In [107]:
print("GPU antes do treino:")
print(
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2,
    ),
    "GB alocados",
)

GPU antes do treino:
0.42 GB alocados


In [108]:
inicio = time.perf_counter()

trainer_benchmark.train()

tempo_benchmark = time.perf_counter() - inicio

memoria_maxima = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print(
    f"Tempo para {MAX_STEPS_BENCHMARK} steps: "
    f"{tempo_benchmark:.2f} segundos"
)

print(
    f"Pico de VRAM: "
    f"{memoria_maxima:.2f} GB"
)

Step,Training Loss
20,2.226630
40,2.182159
60,2.188455
80,2.181635
100,2.181033


Tempo para 100 steps: 32.80 segundos
Pico de VRAM: 2.68 GB


In [109]:
if torch.cuda.is_available():
    memoria_maxima = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

    print(
        f"Pico de VRAM alocada: "
        f"{memoria_maxima:.2f} GB"
    )

Pico de VRAM alocada: 2.68 GB


In [110]:
resultado_avaliacao = trainer.evaluate()

resultado_avaliacao

NameError: name 'trainer' is not defined

In [ ]:
predicao_output = trainer.predict(
    dataset_validacao
)

logits = predicao_output.predictions

predicoes_ids = np.argmax(
    logits,
    axis=-1,
)

predicoes = [
    ID_TO_LABEL[int(pred_id)]
    for pred_id in predicoes_ids
]

In [ ]:
accuracy = accuracy_score(
    validacao["clarity"],
    predicoes,
)

f1_macro = f1_score(
    validacao["clarity"],
    predicoes,
    average="macro",
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {f1_macro:.4f}")

Accuracy: 0.4566
Macro-F1: 0.4436


In [ ]:
labels = ["c1", "c234", "c5"]

matriz = confusion_matrix(
    validacao["clarity"],
    predicoes,
    labels=labels,
)

pd.DataFrame(
    matriz,
    index=[f"Real {label}" for label in labels],
    columns=[f"Predito {label}" for label in labels],
)

,Predito c1,Predito c234,Predito c5
Real c1,580,294,395
Real c234,398,364,609
Real c5,234,254,891


In [ ]:
print(
    classification_report(
        validacao["clarity"],
        predicoes,
        labels=labels,
        digits=4,
    )
)

              precision    recall  f1-score   support

          c1     0.4785    0.4571    0.4676      1269
        c234     0.3991    0.2655    0.3189      1371
          c5     0.4702    0.6461    0.5443      1379

    accuracy                         0.4566      4019
   macro avg     0.4493    0.4562    0.4436      4019
weighted avg     0.4486    0.4566    0.4432      4019

